<img src="../assets/logo.png" width="800">

Made by **Balázs Nagy** and **Márk Domokos** (Adapted for EM/GMM)

[<img src="../assets/open_button.png">](https://colab.research.google.com/github/hubaycsenge/Methods_and_Tools/blob/main/Practice_12/L12%20-%20EM.ipynb)

# Labor 12 - Expectation-Maximization (GMM) Clustering


### 0: Background
While K-Means performs "hard clustering" by assigning each point definitively to a single cluster, Gaussian Mixture Models (GMM) use "soft clustering". The Expectation-Maximization (EM) algorithm fits a mixture of Gaussian distributions to the data, assigning probabilities that a given data point belongs to a specific cluster.

When might we need unsupervised learning?
In cases where we do not know the output, but we are looking for some kind of pattern in our data. Market segmentation, network analysis, or finding overlapping probabilistic distributions in data.

### 1: Import packages

The initial data will be a 2D point cloud. We will also import `multivariate_normal` from `scipy.stats` to calculate Gaussian probabilities easily.

In [ ]:
from scipy.io import loadmat
from scipy.stats import multivariate_normal
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

### 2: Data load

First the data will be downloaded.

In [ ]:
!wget https://github.com/hubaycsenge/Methods_and_Tools/raw/main/assets/Lab11/Lab11data.mat
!wget https://github.com/hubaycsenge/Methods_and_Tools/raw/main/assets/Lab11/bird_small.mat

After the data is downloaded to the local workspace it can be loaded into the notebook.

In [ ]:
data = loadmat("Lab11data.mat")
X = data["X"]
plt.plot(X[:,0],X[:,1], 'yo')
print('X shape:',X.shape)

### 3: EM Initialization
For GMMs, we need to initialize three parameters for each cluster $k$:
1. **Means ($\mu_k$)**: The center of the cluster.
2. **Covariances ($\Sigma_k$)**: The shape/spread of the cluster.
3. **Mixing coefficients ($\pi_k$)**: The prior probability of the cluster (how big the cluster is relative to others).

In [ ]:
def initialize_gmm(X, K):
    n_samples, n_features = X.shape
    
    # Randomly select K data points as initial means
    np.random.seed(42)
    random_indices = np.random.choice(n_samples, K, replace=False)
    mu = X[random_indices]
    
    # Initialize covariances as spherical identity matrices
    sigma = np.array([np.eye(n_features) for _ in range(K)])
    
    # Initialize mixing coefficients uniformly
    pi = np.ones(K) / K
    
    return pi, mu, sigma

K = 3
pi, mu, sigma = initialize_gmm(X, K)
print('Initial means:\n', mu)

### 4: E-Step (Expectation)
Instead of finding the absolute closest centroid, we calculate the "responsibility" ($\gamma$) that cluster $k$ takes for data point $x_i$. This is based on Bayes' theorem.

In [ ]:
def e_step(X, pi, mu, sigma):
    N, D = X.shape
    K = len(pi)
    gamma = np.zeros((N, K))
    
    ################### CODE HERE ######################## 

    #####################################################    
    
    return gamma

gamma = e_step(X, pi, mu, sigma)
print('Responsibilities for the first 3 examples:\n', gamma[0:3])

### 5: M-Step (Maximization)

Once we have the responsibilities (probabilities), we update our parameters ($\pi, \mu, \Sigma$) using weighted averages, where the weights are the responsibilities.

In [ ]:
def m_step(X, gamma):
    N, D = X.shape
    K = gamma.shape[1]
    
    mu = np.zeros((K, D))
    sigma = np.zeros((K, D, D))
    
    ################### CODE HERE ########################

    #####################################################    
    
    return pi, mu, sigma

pi, mu, sigma = m_step(X, gamma)
print("Updated means after one iteration:\n", mu)

### 6: Running the EM Algorithm

We will alternate between the E-step and M-step for a set number of iterations. To visualize, we will plot the means and assign each point to the cluster with the highest responsibility (hard assignment for visualization).

In [ ]:
def plotGMM(X, gamma, mu, it):
    colors = ('b','g','r','c','m','y','k')
    idx = np.argmax(gamma, axis=1) # Hard assign for colors
    K = mu.shape[0]
    
    plt.figure()
    plt.title(f"Iteration {it+1}")
    for i in range(K):
        CL_i = X[np.where(idx == i)[0],:]
        plt.plot(CL_i[:,0],CL_i[:,1], colors[i]+'o', alpha=0.5)
        plt.plot(mu[i,0], mu[i,1], 'kx', markersize=12, markeredgewidth=3)

    plt.show()    

def runEM(X, K, max_iters, plotProgress=False):
    pi, mu, sigma = initialize_gmm(X, K)
    
    for i in range(max_iters):
        if plotProgress:
             print(f'Running the {i+1} iteration of {max_iters}')
        
        gamma = e_step(X, pi, mu, sigma)
        pi, mu, sigma = m_step(X, gamma)
        
        if plotProgress:
            plotGMM(X, gamma, mu, i)
            
    return pi, mu, sigma, gamma

K = 3
max_iters = 5
pi, mu, sigma, gamma = runEM(X, K, max_iters, plotProgress=True)
print("Final GMM Means:\n", mu)

### 7: Problems

Like K-Means, GMMs are heavily dependent on initialization and can get stuck in local optima. However, GMMs are much more flexible because they can model elliptical clusters (thanks to the covariance matrix $\Sigma$), whereas standard K-Means essentially assumes spherical clusters of equal radius.

### 8: EM Clustering on Pixels

We can apply the same EM algorithm for image compression. We will treat each pixel's RGB values as a 3D data point and find Gaussian clusters in color space. Warning: EM is computationally heavier than K-Means, so this might take slightly longer.

In [ ]:
image = loadmat("bird_small.mat")
A_o = image['A']
A = A_o / 255.  # Normalize colors to [0, 1]
img_size = A.shape 
print("Original Image Size:", img_size)
plt.figure()
plt.title("Original Image")
plt.imshow(A)
plt.show()

# Reshape the image into a 2D array of pixels (N x 3)
X2 = A.reshape((int(A.size/3), 3))

K = 4 # Let's use 4 colors for the mixture model
max_iters = 5
print(f"Running EM for {max_iters} iterations on pixels...")
pi_img, mu_img, sigma_img, gamma_img = runEM(X2, K, max_iters, plotProgress=False)
print('EM clustering done...')

### 9: Image compression
To compress the image, we do a hard assignment at the very end. Every pixel is replaced by the mean ($\mu_k$) of the cluster that claims the highest responsibility for it.

In [ ]:
# Get the cluster index with the highest probability for each pixel
idx = np.argmax(gamma_img, axis=1)

# Map each pixel to its cluster's mean color
X_recovered = mu_img[idx, :]
X_recovered = np.reshape(X_recovered, (A.shape[0], A.shape[1], A.shape[2]))

# Ensure values are bounded between 0 and 1 for rendering
X_recovered = np.clip(X_recovered, 0, 1)

plt.figure()
plt.title(f"Compressed Image ({K} colors)")
plt.imshow(X_recovered)
plt.show()